# Data generation

Generates the simulator dataset for one datagen profile. Set `PROFILE` and run every cell. Profiles are
in `datagen/src/erosion_datagen/config.py`; each plan step names the one it needs.

Data generation runs in the `datagen/` environment (its own Python and packages), so this notebook starts
it as a subprocess, shows a progress bar per split, and prints anything else it says. Output goes to
`artifacts/data/<PROFILE>/<split>/`.

**Stopping and resuming.** Interrupt the cell (the stop button) at any time; it stops the whole job,
including the worker processes. Run the cell again to continue: finished runs are kept, and only the
missing ones are simulated. A finished split (it has `meta.json`) is skipped. If the profile's settings
changed since the interruption, that split starts over.


In [ ]:
import json
import os
import signal
import subprocess

from tqdm import tqdm  # the text bar: needs no notebook widgets

from erosion_sim.paths import DATA_DIR, REPO_DIR

PROFILE = "world_master"
SPLITS = ["train", "val", "test"]


def datagen(script: str, bar: tqdm | None = None) -> str:
    """Run Python in the datagen environment and return its output.

    Lines "PROGRESS done total" move ``bar``; other lines are printed. Interrupting stops the whole
    process group (uv, Python, and its worker processes).
    """
    process = subprocess.Popen(["uv", "run", "python", "-u", "-c", script], cwd=REPO_DIR / "datagen",
                               stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, start_new_session=True,
                               env={k: v for k, v in os.environ.items() if k != "VIRTUAL_ENV"})  # datagen has its own
    lines = []
    try:
        for line in process.stdout:
            if line.startswith("PROGRESS ") and bar is not None:
                done, total = map(int, line.split()[1:])
                if bar.total != total:  # first report: count runs already on disk, but not in the rate
                    bar.reset(total=total)
                    bar.update(done)
                    bar.last_print_n, bar.start_t = done, bar._time()
                else:
                    bar.update(done - bar.n)
                continue
            print(line, end="")
            lines.append(line)
    except KeyboardInterrupt:
        os.killpg(process.pid, signal.SIGINT)
        try:
            process.wait(timeout=30)
        except subprocess.TimeoutExpired:
            os.killpg(process.pid, signal.SIGKILL)
        print("stopped; run the cell again to resume")
        raise
    if process.wait():
        raise RuntimeError("datagen failed")
    return "".join(lines)


The profile, and how much disk it will take.

In [ ]:
out = datagen(
    "import json\n"
    "from dataclasses import asdict\n"
    "from erosion_datagen.config import PROFILES\n"
    f"print(json.dumps(asdict(PROFILES[{PROFILE!r}])))\n"
)
cfg = json.loads(out.strip().splitlines()[-1])
for key, value in cfg.items():
    print(f"{key:<18} {value}")
snapshots = cfg["steps"] // cfg["snapshot_every"] + 1
per_run = snapshots * cfg["shape"][0] * cfg["shape"][1] * 4
runs = sum(cfg[f"{s}_runs"] for s in SPLITS)
print(f"\n{runs} runs × {snapshots} snapshots × {cfg['shape'][0]}×{cfg['shape'][1]}: "
      f"{per_run / 1e6:.1f} MB per run, {runs * per_run / 1e9:.1f} GB in all")

## Generate

In [ ]:
for split in SPLITS:
    if (DATA_DIR / PROFILE / split / "meta.json").exists():
        print(f"{split}: already on disk, skipped")
        continue
    with tqdm(desc=split, unit="run", smoothing=0.05) as bar:
        datagen(
            "from erosion_datagen.config import PROFILES\n"
            "from erosion_datagen.simulate import make_dataset\n"
            "def progress(done, total):\n"
            "    print(f'PROGRESS {done} {total}', flush=True)\n"
            f"make_dataset(PROFILES[{PROFILE!r}], {split!r}, progress=progress)\n",
            bar,
        )
    print(f"{split}: done")
